In [1]:
import os, json, math, random, gc, time, ast
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0, resnet18

from tqdm import tqdm

print("\u2705 Imports successful")
print(f"\u2705 CUDA available: {torch.cuda.is_available()}")

✅ Imports successful
✅ CUDA available: False


# BirdCLEF 2026 Inference v7 - 2-Model Ensemble (ResNet18 + EfficientNet-B0)
## Fixes from v6 investigation:
- **Loads ResNet18 folds** (v6 used weak PerchAudio — underfitting)
- **Uniform 0.5 thresholds** (v6 per-species F1 thresholds proven not to generalize)
- **Dynamic windowing** from actual audio duration (correct — same as v6)
- **Matches v7 training** model filenames (`resnet18_v7_fold_*.pt`, `efficientnet_v7_fold_*.pt`)

In [2]:
# === SETUP ===
import os
from pathlib import Path

# Standard Kaggle competition data path
_candidate_dirs = [
    "/kaggle/input/birdclef-2026/test_soundscapes",
    "/kaggle/input/competitions/birdclef-2026/test_soundscapes",
]
TEST_AUDIO_DIR = next((p for p in _candidate_dirs if os.path.isdir(p)), _candidate_dirs[0])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CFG = dict(
    sr=16000,
    n_mels=64,
    n_fft=1024,
    hop=320,
    fmin=60,
    seconds=5,
    batch_size=32,
)

print(f"Device: {device}")
print(f"Test audio dir: {TEST_AUDIO_DIR}")
if not os.path.isdir(TEST_AUDIO_DIR):
    print(f"WARNING: Test data dir not found! Tried: {_candidate_dirs}")
else:
    n_items = len(list(Path(TEST_AUDIO_DIR).iterdir()))
    print(f"Found test directory with {n_items} items")

Device: cpu
Test audio dir: /kaggle/input/competitions/birdclef-2026/test_soundscapes
Found test directory with 1 items


In [3]:
# === LOAD SPECIES & THRESHOLDS ===
# v7 uses uniform 0.5 thresholds — per-species F1 optimization (v6) proven not to generalize
species_paths = [
    "/kaggle/input/datasets/chiragggg/birdclef-2026-v7-model/species.json",
    "/kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species/species.json",
    "species.json",
]

species = None
for path in species_paths:
    try:
        with open(path, "r") as f:
            species = json.load(f)
        print(f"\u2705 Loaded species from: {path}")
        break
    except FileNotFoundError:
        continue

if species is None:
    print("\u274c ERROR: Could not load species.json from any path!")
    raise FileNotFoundError("species.json not found")

# Uniform 0.5 thresholds — try to load from v7 thresholds file, default to 0.5
thresholds_paths = [
    "/kaggle/input/datasets/chiragggg/birdclef-2026-v7-model/optimal_thresholds_v7.json",
    "/kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species/optimal_thresholds_v7.json",
]
optimal_thresholds = None
for path in thresholds_paths:
    try:
        with open(path, "r") as f:
            optimal_thresholds = json.load(f)
        print(f"\u2705 Loaded thresholds from: {path}")
        break
    except FileNotFoundError:
        continue

if optimal_thresholds is None:
    print("\u26a0\ufe0f  No thresholds file found — using uniform 0.5 for all species")
    optimal_thresholds = {sp: 0.5 for sp in species}

idx = {lab: i for i, lab in enumerate(species)}
n_classes = len(species)

print(f"\u2705 Loaded {n_classes} species")
print(f"Threshold sample: {list(optimal_thresholds.items())[:3]}")

✅ Loaded species from: /kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species/species.json
✅ Loaded thresholds from: /kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species/optimal_thresholds_v7.json
✅ Loaded 234 species
Threshold sample: [('1161364', 0.5), ('116570', 0.5), ('1176823', 0.5)]


In [4]:
# === HELPER FUNCTIONS ===
def fixed_length_mono(y, sr, seconds=5):
    target = sr * seconds
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    else:
        y = y[:target]
    return y.astype(np.float32)

def logmel_from_wave(wave, sr):
    S = librosa.feature.melspectrogram(
        y=wave, sr=sr, n_fft=CFG["n_fft"], hop_length=CFG["hop"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=sr // 2, power=2.0
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    S_min = S_db.min()
    S_max = S_db.max()
    if S_max - S_min < 1e-9:
        S_norm = np.zeros_like(S_db, dtype=np.float32)
    else:
        S_norm = (S_db - S_min) / (S_max - S_min + 1e-9)
    return np.clip(S_norm, 0.0, 1.0).astype(np.float32)

print("\u2705 Helper functions defined")

✅ Helper functions defined


In [5]:
# === RESNET18 ARCHITECTURE (Proven 0.648 architecture — replaces v6 PerchAudio) ===
class ResNet18Audio(nn.Module):
    """ResNet18-based classifier for mel spectrograms. Proven architecture (0.648 baseline)."""
    def __init__(self, n_classes: int, n_mels: int = 64):
        super().__init__()
        backbone = resnet18(weights=None)
        backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

print("\u2705 ResNet18Audio model defined")

✅ ResNet18Audio model defined


In [6]:
# === EFFICIENTNET-B0 ARCHITECTURE ===
class EfficientNetB0Audio(nn.Module):
    def __init__(self, n_classes: int, n_mels: int = 64):
        super().__init__()
        self.model = efficientnet_b0(weights=None)
        self.model.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        in_features = self.model.classifier[1].in_features
        self.model.classifier = nn.Identity()
        self.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        features = self.model(x)
        return self.head(features)

print("\u2705 EfficientNetB0Audio model defined")

✅ EfficientNetB0Audio model defined


In [7]:
# === LOAD ALL TRAINED MODELS ===
# v7 model filenames: resnet18_v7_fold_*.pt, efficientnet_v7_fold_*.pt
print("Loading all 10 trained models (5 folds x 2 architectures)...")

# Try v7 model paths first, fall back to v6 filenames for backward compatibility
MODEL_BASE_PATHS = [
    "/kaggle/input/datasets/chiragggg/birdclef-2026-v7-model",
    "/kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species",
]
MODEL_BASE = next((p for p in MODEL_BASE_PATHS if os.path.isdir(p)), MODEL_BASE_PATHS[0])
print(f"Loading models from: {MODEL_BASE}")

resnet_models = []
efficientnet_models = []

for fold_idx in range(5):
    # Load ResNet18 (v7) — fall back to perch_v6 filenames if v7 weights not found
    resnet_path = f"{MODEL_BASE}/resnet18_v7_fold_{fold_idx}.pt"
    if not os.path.exists(resnet_path):
        # Backward compat: try perch v6 filename
        resnet_path = f"{MODEL_BASE}/perch_v6_fold_{fold_idx}.pt"
    resnet_model = ResNet18Audio(n_classes=n_classes, n_mels=CFG["n_mels"]).to(device)
    resnet_model.load_state_dict(torch.load(resnet_path, map_location=device))
    resnet_model.eval()
    resnet_models.append(resnet_model)

    # Load EfficientNet-B0
    eff_path = f"{MODEL_BASE}/efficientnet_v7_fold_{fold_idx}.pt"
    if not os.path.exists(eff_path):
        eff_path = f"{MODEL_BASE}/efficientnet_v6_fold_{fold_idx}.pt"
    eff_model = EfficientNetB0Audio(n_classes=n_classes, n_mels=CFG["n_mels"]).to(device)
    eff_model.load_state_dict(torch.load(eff_path, map_location=device))
    eff_model.eval()
    efficientnet_models.append(eff_model)

print(f"\u2705 Loaded {len(resnet_models)} ResNet18 folds")
print(f"\u2705 Loaded {len(efficientnet_models)} EfficientNet-B0 folds")

Loading all 10 trained models (5 folds x 2 architectures)...
Loading models from: /kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species
✅ Loaded 5 ResNet18 folds
✅ Loaded 5 EfficientNet-B0 folds


In [8]:
# === TEST AUDIO DATASET (DYNAMIC WINDOWS, 5s each) ===
# Loads the full audio file and creates one 5-second segment per window.
# Window count is inferred from the actual file duration.
# e.g. 60-second file -> 12 windows (_5, _10, ..., _60)

class TestAudioDataset(Dataset):
    def __init__(self, audio_path, cfg):
        sr = cfg["sr"]
        seg_samples = sr * cfg["seconds"]   # 16000 * 5 = 80000 samples

        try:
            y, sr0 = sf.read(audio_path, always_2d=False)
            if y.ndim == 2:
                y = y.mean(axis=1)
            if sr0 != sr:
                y = librosa.resample(y, orig_sr=sr0, target_sr=sr)
            y = y.astype(np.float32)
        except Exception as e:
            print(f"Warning: failed to load {audio_path}: {e}")
            y = np.zeros(seg_samples, dtype=np.float32)

        # How many complete 5-second windows fit in this file?
        n_full = len(y) // seg_samples
        self.n_windows = max(n_full, 1)

        # Trim/pad to exact multiple of segment length
        target_len = self.n_windows * seg_samples
        if len(y) < target_len:
            y = np.pad(y, (0, target_len - len(y)))
        else:
            y = y[:target_len]

        self.y = y
        self.sr = sr
        self.seg_samples = seg_samples

    def __len__(self):
        return self.n_windows

    def __getitem__(self, idx):
        start = idx * self.seg_samples
        segment = self.y[start: start + self.seg_samples].copy()
        if segment.dtype != np.float32:
            segment = segment.astype(np.float32)
        mel = logmel_from_wave(segment, self.sr)
        return torch.from_numpy(np.expand_dims(mel, 0)).float()

print("TestAudioDataset: dynamic windows from actual audio duration")

TestAudioDataset: dynamic windows from actual audio duration


In [9]:
# === PREDICTION FUNCTION ===
def get_predictions_for_audio(audio_path):
    """
    Get ensemble predictions from all models, one prediction per 5-second window.
    Window count is determined by the actual audio duration (e.g. 60s = 12 windows).
    Returns a list of numpy arrays, one per window.
    """
    dataset = TestAudioDataset(audio_path, CFG)
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)

    window_predictions = []

    with torch.no_grad():
        for x in loader:
            x = x.to(device)
            window_preds = np.zeros((n_classes,), dtype=np.float32)

            # ResNet18 ensemble
            for model in resnet_models:
                probs = torch.sigmoid(model(x)).cpu().numpy()[0]
                window_preds += probs

            # EfficientNet-B0 ensemble
            for model in efficientnet_models:
                probs = torch.sigmoid(model(x)).cpu().numpy()[0]
                window_preds += probs

            # Average over 10 models (5 ResNet18 + 5 EfficientNet)
            window_preds /= (len(resnet_models) + len(efficientnet_models))
            window_predictions.append(window_preds)

    return window_predictions

print("Prediction function defined")

Prediction function defined


In [10]:
# === GENERATE PREDICTIONS ===
# Evaluation metric: macro-averaged ROC-AUC -> output raw probabilities (no thresholding)
test_dir = Path(TEST_AUDIO_DIR)

# Search recursively for all common audio formats
test_files = []
if test_dir.exists():
    for pattern in ["*.ogg", "*.wav", "*.mp3", "*.flac", "*.m4a"]:
        test_files.extend(test_dir.rglob(pattern))
    test_files = sorted(set(test_files))

print(f"Found {len(test_files)} test files in {TEST_AUDIO_DIR}")
if test_files:
    from collections import Counter
    fmts = Counter(f.suffix.lower() for f in test_files)
    for fmt, count in sorted(fmts.items()):
        print(f"  {fmt}: {count} files")
else:
    print("WARNING: No test audio files found!")
    print(f"  Tried path: {TEST_AUDIO_DIR}")

predictions = {}

for audio_path in tqdm(test_files, desc="Predicting"):
    stem = audio_path.stem
    try:
        window_preds_list = get_predictions_for_audio(str(audio_path))
        n_windows = len(window_preds_list)
        # Row ID suffix = end time of each 5-second window: _5, _10, _15, ...
        window_offsets = [(i + 1) * CFG["seconds"] for i in range(n_windows)]
        for window_idx, window_preds in enumerate(window_preds_list):
            row_id = f"{stem}_{window_offsets[window_idx]}"
            predictions[row_id] = window_preds
    except Exception as e:
        print(f"Error with {audio_path.name}: {e}")
        # Fallback: fill first 3 windows with zeros
        for offset in [5, 10, 15]:
            predictions[f"{stem}_{offset}"] = np.zeros(n_classes, dtype=np.float32)

print(f"\nGenerated {len(predictions)} rows from {len(test_files)} files")
if test_files:
    print(f"Average windows per file: {len(predictions)/len(test_files):.1f}")

Found 0 test files in /kaggle/input/competitions/birdclef-2026/test_soundscapes
  Tried path: /kaggle/input/competitions/birdclef-2026/test_soundscapes


Predicting: 0it [00:00, ?it/s]


Generated 0 rows from 0 files


In [11]:
# === CHECK TEST METADATA ===
print("Checking test metadata...")
test_meta = None
try:
    test_meta = pd.read_csv("/kaggle/input/competitions/birdclef-2026/test.csv")
    print(f"\u2705 Test metadata shape: {test_meta.shape}")
    print(f"Test metadata columns: {test_meta.columns.tolist()}")
    print(f"\nFirst 10 rows:")
    print(test_meta.head(10))
except FileNotFoundError:
    print("\u26a0\ufe0f  No test.csv found - checking alternative paths...")
    import glob
    csv_files = glob.glob("/kaggle/input/competitions/birdclef-2026/*.csv")
    print(f"Available CSV files: {csv_files}")

Checking test metadata...
⚠️  No test.csv found - checking alternative paths...
Available CSV files: ['/kaggle/input/competitions/birdclef-2026/sample_submission.csv', '/kaggle/input/competitions/birdclef-2026/taxonomy.csv', '/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv', '/kaggle/input/competitions/birdclef-2026/train.csv']


In [12]:
# === CREATE SUBMISSION (Uniform 0.5 Thresholds) ===
# v7 uses uniform 0.5 thresholds — NOT per-species F1 optimization.
# Per-species F1 thresholds (v5/v6) were proven not to generalize to test soundscapes.

# Load sample submission to get authoritative column order AND all expected row_ids.
sample_sub_paths = [
    "/kaggle/input/birdclef-2026/sample_submission.csv",
    "/kaggle/input/competitions/birdclef-2026/sample_submission.csv",
]
sample_sub = None
for path in sample_sub_paths:
    try:
        sample_sub = pd.read_csv(path)
        print(f"Loaded sample_submission.csv from: {path}")
        print(f"Shape: {sample_sub.shape}")
        break
    except FileNotFoundError:
        continue

if sample_sub is None:
    raise FileNotFoundError(
        f"sample_submission.csv not found! Tried: {sample_sub_paths}"
    )

# Authoritative column list from Kaggle: ['row_id'] + species in correct order
SUBMIT_COLS = list(sample_sub.columns)
submit_species = SUBMIT_COLS[1:]
print(f"Expected columns: {len(SUBMIT_COLS)} (1 row_id + {len(submit_species)} species)")
print(f"Expected rows in template: {len(sample_sub)}")

# Map species name -> index in model output
species_to_idx = {sp: i for i, sp in enumerate(species)}
missing = [s for s in submit_species if s not in species_to_idx]
print(f"Model has {len(species)} species, submission needs {len(submit_species)}")
print(f"Species missing from model (will be 0.0): {len(missing)}")

matched = sum(1 for rid in sample_sub["row_id"] if rid in predictions)
print(f"\nPredictions matched to template row_ids: {matched} / {len(sample_sub)}")
if matched < len(sample_sub):
    unmatched = [rid for rid in sample_sub["row_id"] if rid not in predictions][:5]
    print(f"  Sample unmatched: {unmatched}")

# Build submission by iterating over the TEMPLATE rows
# Guarantees exactly the rows Kaggle expects, with 0.0 for any missing predictions.
rows = []
for row_id in sample_sub["row_id"]:
    preds = predictions.get(row_id, None)
    row = {"row_id": row_id}
    for sp in submit_species:
        if preds is not None and sp in species_to_idx:
            row[sp] = float(preds[species_to_idx[sp]])
        else:
            row[sp] = 0.0   # Species not in model get 0.0 (correct: not present)
    rows.append(row)

submission_df = pd.DataFrame(rows, columns=SUBMIT_COLS)
submission_df["row_id"] = submission_df["row_id"].astype(str)
for col in submit_species:
    submission_df[col] = submission_df[col].astype("float64")
submission_df = submission_df.fillna(0.0)

print(f"\nsubmission_df created:")
print(f"  Rows: {len(submission_df)}")
print(f"  Cols: {len(submission_df.columns)}")
print(f"  row_id dtype: {submission_df['row_id'].dtype}")
print(f"  species dtype: {submission_df[submit_species[0]].dtype}")
print(f"\nFirst 3 row_ids: {submission_df['row_id'].head(3).tolist()}")

Loaded sample_submission.csv from: /kaggle/input/competitions/birdclef-2026/sample_submission.csv
Shape: (3, 235)
Expected columns: 235 (1 row_id + 234 species)
Expected rows in template: 3
Model has 234 species, submission needs 234
Species missing from model (will be 0.0): 0

Predictions matched to template row_ids: 0 / 3
  Sample unmatched: ['BC2026_Test_0001_S05_20250227_010002_5', 'BC2026_Test_0001_S05_20250227_010002_10', 'BC2026_Test_0001_S05_20250227_010002_15']

submission_df created:
  Rows: 3
  Cols: 235
  row_id dtype: object
  species dtype: float64

First 3 row_ids: ['BC2026_Test_0001_S05_20250227_010002_5', 'BC2026_Test_0001_S05_20250227_010002_10', 'BC2026_Test_0001_S05_20250227_010002_15']


In [13]:
# === SAVE SUBMISSION ===
submission_df.to_csv("/kaggle/working/submission.csv", index=False)

print(f"\u2705 Submission saved to: /kaggle/working/submission.csv")
print(f"\n\U0001f4ca SUBMISSION SUMMARY:")
print(f"  Total predictions: {len(submission_df)}")
species_cols = [col for col in submission_df.columns if col != "row_id"]
print(f"  Total species: {len(species_cols)}")
if len(submission_df) > 0:
    print(f"  Avg probability per species: {submission_df[species_cols].mean().mean():.4f}")
print(f"\n\u2705 INFERENCE COMPLETE!")
print(f"\n\U0001f4c8 ENSEMBLE STRATEGY (v7):")
print(f"  - 5 ResNet18 folds (proven architecture, replaces weak v6 PerchAudio)")
print(f"  - 5 EfficientNet-B0 folds")
print(f"  - Dynamic windowing (one row per 5-second segment from actual audio duration)")
print(f"  - Uniform 0.5 thresholds (not per-species F1 — proven not to generalize)")
print(f"  - 10 model average per window (5 folds x 2 architectures)")

✅ Submission saved to: /kaggle/working/submission.csv

📊 SUBMISSION SUMMARY:
  Total predictions: 3
  Total species: 234
  Avg probability per species: 0.0000

✅ INFERENCE COMPLETE!

📈 ENSEMBLE STRATEGY (v7):
  - 5 ResNet18 folds (proven architecture, replaces weak v6 PerchAudio)
  - 5 EfficientNet-B0 folds
  - Dynamic windowing (one row per 5-second segment from actual audio duration)
  - Uniform 0.5 thresholds (not per-species F1 — proven not to generalize)
  - 10 model average per window (5 folds x 2 architectures)


In [14]:
# === VALIDATE SUBMISSION FORMAT ===
print("Validating submission format...")

# Check for NaN values
nan_count = submission_df.isna().sum().sum()
if nan_count > 0:
    print(f"\u26a0\ufe0f  WARNING: Found {nan_count} NaN values in submission!")
else:
    print("\u2705 No NaN values")

# Check data types
wrong_dtypes = []
for col in submission_df.columns:
    expected = "object" if col == "row_id" else "float64"
    if str(submission_df[col].dtype) != expected:
        wrong_dtypes.append((col, submission_df[col].dtype, expected))

if wrong_dtypes:
    print(f"\u26a0\ufe0f  WARNING: {len(wrong_dtypes)} columns have wrong dtype!")
    for col, actual, expected in wrong_dtypes[:5]:
        print(f"  {col}: {actual} (expected {expected})")
else:
    print("\u2705 All dtypes correct")

# Check value ranges for species columns
out_of_range = False
for col in submission_df.columns[1:]:
    min_val = submission_df[col].min()
    max_val = submission_df[col].max()
    if min_val < 0 or max_val > 1:
        print(f"\u26a0\ufe0f  WARNING: {col} has values outside [0,1]: min={min_val}, max={max_val}")
        out_of_range = True
        break
if not out_of_range:
    print("\u2705 All species values in [0, 1]")

print(f"\n\u2705 Validation complete — ready to submit!")

Validating submission format...
✅ No NaN values
✅ All dtypes correct
✅ All species values in [0, 1]

✅ Validation complete — ready to submit!
